# 1. Environment Setup

In [8]:
# ============================================================
# CELL 1 — Environment Setup
# ============================================================

import os, subprocess, sys, time

print("🧹 Cleaning environment...")
subprocess.run("fuser -k 8000/tcp 2>/dev/null || true", shell=True)
time.sleep(2)

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q",
     "torch", "torchvision", "torchaudio", "transformers", "accelerate", "bitsandbytes"],
    stderr=subprocess.DEVNULL
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
print("✅ Environment cleaned")

install_groups = [
    ("PyTorch (CUDA 12.1)", ["--index-url", "https://download.pytorch.org/whl/cu121",
        "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1"]),
    ("Transformers stack", ["transformers==4.46.3", "accelerate==0.34.2",
        "bitsandbytes==0.43.3", "peft==0.13.2", "sentencepiece", "einops"]),
    ("Vector / embeddings", ["sentence-transformers==2.7.0", "faiss-cpu==1.8.0"]),
    ("Audio processing", ["faster-whisper==1.0.3", "webrtcvad==2.0.10", "pydub", "soundfile"]),
    ("Document processing", ["PyMuPDF==1.24.0", "python-docx==1.1.2", "python-pptx==0.6.23",
        "mammoth==1.8.0", "Pillow==10.4.0", "pytesseract==0.3.13",
        "beautifulsoup4==4.12.3", "lxml==5.3.0", "markdown==3.7"]),
    ("Backend framework", ["fastapi==0.115.0", "uvicorn[standard]==0.32.0",
        "python-socketio==5.11.0", "aiofiles==24.1.0",
        "python-multipart==0.0.12", "httpx==0.27.2"]),
    ("Database & auth", ["supabase==2.9.0", "pgvector==0.3.5", "pyjwt==2.9.0",
        "bcrypt==4.2.0", "python-dotenv==1.0.1"]),
    ("Utilities", ["pyngrok==7.2.0", "nest_asyncio==1.6.0", "orjson==3.10.7",
        "python-dateutil==2.9.0", "tenacity==9.0.0",
        "numpy", "pandas", "scikit-learn"]),
]

for name, pkgs in install_groups:
    print(f"📦 Installing {name}...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ⚠️  {name}:\n{r.stderr[-400:]}")
    else:
        print(f"  ✅ OK")

import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}  {p.total_memory/1e9:.1f} GB")
print("\n✅ Cell 1 complete")

🧹 Cleaning environment...
✅ Environment cleaned
📦 Installing PyTorch (CUDA 12.1)...
  ✅ OK
📦 Installing Transformers stack...
  ✅ OK
📦 Installing Vector / embeddings...
  ✅ OK
📦 Installing Audio processing...
  ✅ OK
📦 Installing Document processing...
  ✅ OK
📦 Installing Backend framework...
  ✅ OK
📦 Installing Database & auth...
  ✅ OK
📦 Installing Utilities...
  ✅ OK

PyTorch : 2.4.1+cu121
CUDA    : True
GPU     : Tesla T4  15.6 GB

✅ Cell 1 complete


# 2. Configuration & Project Structure

In [9]:
# ============================================================
# CELL 2 — Configuration & Structured Logging Setup
# ============================================================

import os, json, logging, secrets
from pathlib import Path
from datetime import datetime
from google.colab import userdata

# ── Structured logging ────────────────────────────────────────
LOG_FORMAT = "%(asctime)s [%(name)-18s] %(levelname)-8s %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
    ]
)
# Silence noisy third-party loggers
for noisy in ("httpx", "httpcore", "urllib3", "filelock", "transformers.tokenization"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

log_cfg = logging.getLogger("config")

# ── Secrets ───────────────────────────────────────────────────
def get_secret(key: str, required: bool = True, default: str = "") -> str:
    try:
        v = userdata.get(key)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(key, "")
    if v:
        return v
    if required:
        raise RuntimeError(
            f"❌ Required secret '{key}' missing. Add it in the 🔑 Secrets panel."
        )
    return default

SUPABASE_URL         = get_secret("SUPABASE_URL")
SUPABASE_KEY         = get_secret("SUPABASE_KEY")
SUPABASE_SERVICE_KEY = get_secret("SUPABASE_SERVICE_KEY")
NGROK_AUTH_TOKEN     = get_secret("NGROK_AUTH_TOKEN")

JWT_SECRET = get_secret("JWT_SECRET", required=False, default="")
if not JWT_SECRET:
    JWT_SECRET = secrets.token_hex(32)
    log_cfg.warning("JWT_SECRET not set — random one generated (tokens reset on restart).")

os.environ.update({
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_KEY": SUPABASE_KEY,
    "SUPABASE_SERVICE_KEY": SUPABASE_SERVICE_KEY,
    "JWT_SECRET": JWT_SECRET,
    "NGROK_AUTH_TOKEN": NGROK_AUTH_TOKEN,
})
log_cfg.info("Secrets loaded ✅")

# ── Project directories ───────────────────────────────────────
PROJECT_ROOT = Path("/content/smart_classroom")
DIRS = {
    "uploads":          PROJECT_ROOT / "uploads",
    "cache":            PROJECT_ROOT / "cache",
    "audio":            PROJECT_ROOT / "audio",
    "exports":          PROJECT_ROOT / "exports",
    "temp":             PROJECT_ROOT / "temp",
    "models":           PROJECT_ROOT / "models",
    "embeddings":       PROJECT_ROOT / "embeddings",
    "student_profiles": PROJECT_ROOT / "student_profiles",
    "analytics":        PROJECT_ROOT / "analytics",
    "logs":             PROJECT_ROOT / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Add file handler now that log dir exists
_fh = logging.FileHandler(DIRS["logs"] / "classroom.log", encoding="utf-8")
_fh.setFormatter(logging.Formatter(LOG_FORMAT, datefmt="%H:%M:%S"))
logging.getLogger().addHandler(_fh)

with open(PROJECT_ROOT / "config.json", "w") as f:
    json.dump({
        "project_root": str(PROJECT_ROOT),
        "directories": {k: str(v) for k, v in DIRS.items()},
        "created_at": datetime.utcnow().isoformat(),
    }, f, indent=2)

log_cfg.info("Project root: %s", PROJECT_ROOT)
print("✅ Cell 2 complete")

✅ Cell 2 complete


/tmp/ipykernel_23863/3228416912.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


# 3. Database System

In [10]:
import sqlite3, threading, json, uuid
from typing import Optional, List, Dict, Any
from supabase import create_client, Client

log_db = logging.getLogger("database")


class SupabaseDB:
    """Supabase (Postgres + pgvector) remote database."""

    def __init__(self):
        self.client: Optional[Client] = None
        self._connected = False
        self._lock = threading.Lock()

    def connect(self) -> bool:
        with self._lock:
            if self._connected:
                return True
            try:
                self.client = create_client(
                    os.environ["SUPABASE_URL"],
                    os.environ["SUPABASE_SERVICE_KEY"],   # service key bypasses RLS
                )
                self.client.table("users").select("id").limit(1).execute()
                self._connected = True
                log_db.info("Supabase connected ✅")
                return True
            except Exception as e:
                log_db.error("Supabase connection failed: %s", e)
                return False

    @property
    def connected(self) -> bool:
        return self._connected

    def _c(self) -> Client:
        if not self._connected:
            raise RuntimeError("Supabase not connected — call connect() first")
        return self.client

    # ── Users ────────────────────────────────────────────────

    def create_user(
        self, user_id: str, username: str, email: str,
        password_hash: str, role: str, profile: dict = None
    ) -> bool:
        """
        FIX #1: 'id' is now explicitly included so the Python-generated
        UUID is stored rather than a DB-auto-generated one.
        """
        try:
            data = {
                "id":            user_id,          # ← was missing; caused uid mismatch
                "username":      username,
                "email":         email,
                "password_hash": password_hash,
                "role":          role,
                "profile":       profile or {},
                "created_at":    datetime.utcnow().isoformat(),
            }
            self._c().table("users").insert(data).execute()
            log_db.info("User created: %s (%s)", username, role)
            return True
        except Exception as e:
            log_db.error("create_user(%s): %s", username, e)
            return False

    def get_user(self, username: str) -> Optional[Dict]:
        try:
            r = self._c().table("users").select("*").eq("username", username).execute()
            return r.data[0] if r.data else None
        except Exception as e:
            log_db.error("get_user(%s): %s", username, e)
            return None

    def get_user_by_id(self, user_id: str) -> Optional[Dict]:
        try:
            r = self._c().table("users").select("*").eq("id", user_id).execute()
            return r.data[0] if r.data else None
        except Exception as e:
            log_db.error("get_user_by_id(%s): %s", user_id, e)
            return None

    def update_last_login(self, user_id: str) -> bool:
        """
        FIX #2: This method was called in AuthManager.authenticate() but never defined.
        """
        try:
            self._c().table("users").update({
                "last_login": datetime.utcnow().isoformat()
            }).eq("id", user_id).execute()
            log_db.debug("last_login updated for %s", user_id)
            return True
        except Exception as e:
            log_db.warning("update_last_login(%s): %s", user_id, e)
            return False

    # ── Lectures ─────────────────────────────────────────────

    def create_lecture(
        self, title: str, course_id: Optional[str], uploaded_by: Optional[str],
        file_type: str, file_path: str, metadata: dict = None
    ) -> Optional[str]:
        """
        FIX #4: course_id and uploaded_by were accepted as params
        but never included in the insert dict.
        """
        try:
            data = {
                "title":       title,
                "course_id":   course_id,        # ← was missing
                "uploaded_by": uploaded_by,      # ← was missing
                "file_type":   file_type,
                "file_path":   file_path,
                "metadata":    metadata or {},
                "created_at":  datetime.utcnow().isoformat(),
            }
            r = self._c().table("lectures").insert(data).execute()
            lecture_id = r.data[0]["id"] if r.data else None
            if lecture_id:
                log_db.info("Lecture created: %s → %s", title, lecture_id)
            return lecture_id
        except Exception as e:
            log_db.error("create_lecture(%s): %s", title, e)
            return None

    def get_lectures(self, course_id: Optional[str] = None, limit: int = 50) -> List[Dict]:
        try:
            q = self._c().table("lectures").select(
                "id, title, course_id, uploaded_by, file_type, metadata, created_at"
            ).order("created_at", desc=True).limit(limit)
            if course_id:
                q = q.eq("course_id", course_id)
            return q.execute().data or []
        except Exception as e:
            log_db.error("get_lectures: %s", e)
            return []

    # ── Embeddings ────────────────────────────────────────────

    def store_embeddings_batch(self, rows: List[Dict]) -> int:
        if not rows:
            return 0
        try:
            self._c().table("lecture_chunks").insert(rows).execute()
            log_db.debug("Stored %d embedding chunks", len(rows))
            return len(rows)
        except Exception as e:
            log_db.error("store_embeddings_batch (%d rows): %s", len(rows), e)
            return 0

    def search_similar(
        self, query_embedding: List[float],
        lecture_id: Optional[str] = None,
        top_k: int = 5
    ) -> List[Dict]:
        try:
            params: Dict[str, Any] = {
                "query_embedding": query_embedding,
                "match_count":     top_k,
            }
            if lecture_id:
                params["lecture_filter"] = lecture_id
            r = self._c().rpc("match_documents", params).execute()
            return r.data or []
        except Exception as e:
            log_db.error("search_similar: %s", e)
            return []

    # ── Transcripts ───────────────────────────────────────────

    def bulk_insert_transcripts(self, records: List[Dict]) -> bool:
        if not records:
            return True
        try:
            self._c().table("live_transcripts").insert(records).execute()
            log_db.debug("Synced %d transcripts", len(records))
            return True
        except Exception as e:
            log_db.error("bulk_insert_transcripts (%d): %s", len(records), e)
            return False

    # ── Sessions ──────────────────────────────────────────────

    def create_session_record(
        self, session_id: str, title: str,
        teacher_id: Optional[str] = None, course_id: Optional[str] = None
    ) -> bool:
        try:
            self._c().table("sessions").insert({
                "session_id": session_id,
                "title":      title,
                "teacher_id": teacher_id,
                "course_id":  course_id,
                "started_at": datetime.utcnow().isoformat(),
                "status":     "active",
            }).execute()
            return True
        except Exception as e:
            log_db.error("create_session_record(%s): %s", session_id, e)
            return False

    def end_session_record(self, session_id: str) -> bool:
        try:
            self._c().table("sessions").update({
                "ended_at": datetime.utcnow().isoformat(),
                "status":   "ended",
            }).eq("session_id", session_id).execute()
            return True
        except Exception as e:
            log_db.error("end_session_record(%s): %s", session_id, e)
            return False

    # ── Quiz ──────────────────────────────────────────────────

    def save_quiz_result(
        self, user_id: str, lecture_id: Optional[str],
        questions: list, answers: list, score: float, difficulty: str = "medium"
    ) -> bool:
        try:
            self._c().table("quiz_results").insert({
                "user_id":      user_id,
                "lecture_id":   lecture_id,
                "questions":    questions,
                "answers":      answers,
                "score":        round(score, 2),
                "difficulty":   difficulty,
                "completed_at": datetime.utcnow().isoformat(),
            }).execute()
            return True
        except Exception as e:
            log_db.error("save_quiz_result(%s): %s", user_id, e)
            return False


# ── Local SQLite cache ────────────────────────────────────────

class LocalCache:
    """Fast SQLite cache for session state + offline buffering."""

    def __init__(self, db_path: Path):
        self._conn = sqlite3.connect(
            str(db_path), check_same_thread=False, isolation_level=None
        )
        self._conn.row_factory = sqlite3.Row
        self._conn.execute("PRAGMA journal_mode=WAL")
        self._conn.execute("PRAGMA synchronous=NORMAL")
        self._lock = threading.Lock()
        self._create_tables()
        log_db.info("LocalCache ready: %s", db_path)

    def _create_tables(self):
        with self._lock:
            self._conn.executescript("""
                BEGIN;
                CREATE TABLE IF NOT EXISTS active_sessions (
                    session_id TEXT PRIMARY KEY,
                    title      TEXT NOT NULL,
                    started_at TEXT NOT NULL,
                    ended_at   TEXT,
                    status     TEXT NOT NULL DEFAULT 'active'
                );
                CREATE TABLE IF NOT EXISTS transcript_buffer (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT    NOT NULL,
                    speaker    TEXT    NOT NULL,
                    text       TEXT    NOT NULL,
                    timestamp  REAL    NOT NULL,
                    synced     INTEGER NOT NULL DEFAULT 0
                );
                CREATE TABLE IF NOT EXISTS student_profiles (
                    user_id        TEXT PRIMARY KEY,
                    level          TEXT NOT NULL DEFAULT 'beginner',
                    weak_topics    TEXT NOT NULL DEFAULT '[]',
                    strong_topics  TEXT NOT NULL DEFAULT '[]',
                    learning_style TEXT NOT NULL DEFAULT 'visual',
                    last_updated   TEXT NOT NULL
                );
                COMMIT;
            """)

    def create_session(self, session_id: str, title: str):
        with self._lock:
            self._conn.execute(
                "INSERT OR REPLACE INTO active_sessions "
                "(session_id, title, started_at, status) VALUES (?,?,?,?)",
                (session_id, title, datetime.utcnow().isoformat(), "active"),
            )

    def end_session(self, session_id: str):
        """
        FIX #3: This method was called in LiveTranscriptionEngine.end_session()
        but was never defined in LocalCache.
        """
        with self._lock:
            self._conn.execute(
                "UPDATE active_sessions SET status=?, ended_at=? WHERE session_id=?",
                ("ended", datetime.utcnow().isoformat(), session_id),
            )
        log_db.debug("LocalCache session ended: %s", session_id)

    def cache_transcript(self, session_id: str, speaker: str, text: str, timestamp: float):
        with self._lock:
            self._conn.execute(
                "INSERT INTO transcript_buffer (session_id, speaker, text, timestamp) "
                "VALUES (?,?,?,?)",
                (session_id, speaker, text, timestamp),
            )

    def pop_unsynced_transcripts(self, limit: int = 200) -> List[Dict]:
        with self._lock:
            cur = self._conn.execute(
                "SELECT * FROM transcript_buffer WHERE synced=0 LIMIT ?", (limit,)
            )
            rows = [dict(r) for r in cur.fetchall()]
            if rows:
                ids = [(r["id"],) for r in rows]
                self._conn.executemany(
                    "UPDATE transcript_buffer SET synced=1 WHERE id=?", ids
                )
            return rows

    def update_student_profile(
        self, user_id: str, level: str = "beginner",
        weak_topics: list = None, strong_topics: list = None,
        learning_style: str = "visual"
    ):
        with self._lock:
            self._conn.execute(
                "INSERT OR REPLACE INTO student_profiles "
                "(user_id, level, weak_topics, strong_topics, learning_style, last_updated) "
                "VALUES (?,?,?,?,?,?)",
                (
                    user_id, level,
                    json.dumps(weak_topics or []),
                    json.dumps(strong_topics or []),
                    learning_style,
                    datetime.utcnow().isoformat(),
                ),
            )

    def get_active_sessions(self) -> List[Dict]:
        with self._lock:
            cur = self._conn.execute(
                "SELECT * FROM active_sessions WHERE status='active'"
            )
            return [dict(r) for r in cur.fetchall()]


# ── Initialise ────────────────────────────────────────────────
log_db.info("Connecting to databases...")
supabase_db = SupabaseDB()
if not supabase_db.connect():
    log_db.warning("Supabase connection failed — check secrets.")
local_cache = LocalCache(DIRS["cache"] / "classroom.db")
print("✅ Cell 3 complete — databases initialised")

✅ Cell 3 complete — databases initialised


# 4. Authentication Profiles

In [11]:
import secrets as py_secrets
from datetime import timedelta, timezone
from typing import Tuple
import bcrypt, jwt
from jwt import ExpiredSignatureError, InvalidTokenError

log_auth = logging.getLogger("auth")
VALID_ROLES  = frozenset({"teacher", "student", "admin"})
TOKEN_EXPIRY = timedelta(hours=24)


class TokenResult:
    __slots__ = ("payload", "error", "expired")

    def __init__(self, payload=None, error=None, expired=False):
        self.payload = payload
        self.error   = error
        self.expired = expired

    @property
    def ok(self) -> bool:
        return self.payload is not None


class AuthManager:
    def __init__(self, db: SupabaseDB, cache: LocalCache):
        self.db    = db
        self.cache = cache
        self.secret = os.environ["JWT_SECRET"]

    @staticmethod
    def hash_password(pw: str) -> str:
        return bcrypt.hashpw(pw.encode(), bcrypt.gensalt(12)).decode()

    @staticmethod
    def verify_password(pw: str, hashed: str) -> bool:
        try:
            return bcrypt.checkpw(pw.encode(), hashed.encode())
        except Exception:
            return False

    def register_user(
        self, username: str, email: str, password: str,
        role: str, learning_style: str = "visual"
    ) -> Tuple[Optional[str], Optional[str]]:
        if role not in VALID_ROLES:
            return None, f"Invalid role '{role}'."
        if len(password) < 8:
            return None, "Password must be ≥ 8 characters."
        if self.db.get_user(username):
            return None, f"Username '{username}' already taken."

        uid = str(uuid.uuid4())          # deterministic UUID
        ok = self.db.create_user(
            uid, username, email,
            self.hash_password(password), role,
            {"learning_style": learning_style, "level": "beginner", "preferences": {}},
        )
        if not ok:
            return None, "Database error — user could not be created."

        if role == "student":
            self.cache.update_student_profile(uid, "beginner", [], [], learning_style)

        log_auth.info("Registered user: %s (%s) → %s", username, role, uid)
        # FIX #6: return the uid we actually stored (not a separate one)
        return uid, None

    def authenticate(
        self, username: str, password: str
    ) -> Tuple[Optional[Dict], Optional[str]]:
        user = self.db.get_user(username)
        # Constant-time check (prevents username enumeration via timing)
        dummy_hash = "$2b$12$invalidhashfortimingneutralityonly....................."
        candidate  = user["password_hash"] if user else dummy_hash

        if not self.verify_password(password, candidate) or not user:
            log_auth.warning("Failed login attempt for username='%s'", username)
            return None, "Invalid username or password."

        # FIX #5: update_last_login now exists; errors are non-fatal
        self.db.update_last_login(user["id"])

        log_auth.info("Login: %s (%s)", username, user["role"])
        return {
            "id":       user["id"],
            "username": user["username"],
            "email":    user["email"],
            "role":     user["role"],
            "profile":  user.get("profile", {}),
        }, None

    def generate_token(self, user_data: Dict) -> str:
        now = datetime.now(tz=timezone.utc)
        return jwt.encode(
            {
                "user_id":  user_data["id"],
                "username": user_data["username"],
                "role":     user_data["role"],
                "iat":      now,
                "exp":      now + TOKEN_EXPIRY,
            },
            self.secret,
            algorithm="HS256",
        )

    def verify_token(self, token: str) -> TokenResult:
        try:
            payload = jwt.decode(token, self.secret, algorithms=["HS256"])
            return TokenResult(payload=payload)
        except ExpiredSignatureError:
            return TokenResult(error="Token expired.", expired=True)
        except InvalidTokenError as e:
            return TokenResult(error=f"Invalid token: {e}")
        except Exception as e:
            log_auth.warning("Token verify error: %s", e)
            return TokenResult(error="Token verification failed.")


auth_manager = AuthManager(supabase_db, local_cache)

# Create default admin if missing
if not supabase_db.get_user("admin"):
    uid, err = auth_manager.register_user(
        "admin", "admin@classroom.ai", "Admin1234!", "admin"
    )
    if uid:
        print("✅ Default admin created (admin / Admin1234!)  ⚠️  Change the password!")
    else:
        log_auth.error("Admin creation failed: %s", err)

print("✅ Cell 4 complete — auth ready")

✅ Cell 4 complete — auth ready


# 5. File Processor

In [12]:
import base64
from io import BytesIO
import fitz, docx, mammoth, markdown
from PIL import Image
from pptx import Presentation

log_fp = logging.getLogger("file_processor")
MAX_IMAGE_PIXELS   = (640, 640)
MAX_IMAGES_PER_DOC = 30


class FileProcessor:
    SUPPORTED_FORMATS = {
        ".pdf":  "PDF",
        ".docx": "Word",
        ".pptx": "PowerPoint",
        ".txt":  "Text",
        ".md":   "Markdown",
    }

    def process_file(self, file_path: Path) -> Dict:
        suffix = file_path.suffix.lower()
        if suffix not in self.SUPPORTED_FORMATS:
            return self._err(f"Unsupported format: '{suffix}'")
        log_fp.info("Processing %s (%s)", file_path.name, suffix)
        try:
            dispatcher = {".pdf": self._pdf, ".docx": self._docx, ".pptx": self._pptx}
            handler    = dispatcher.get(suffix, self._text)
            result     = handler(file_path)
            if result["success"]:
                log_fp.info(
                    "Processed %s — %d chars, %d images",
                    file_path.name,
                    len(result["content"]["text"]),
                    result["content"]["image_count"],
                )
            return result
        except Exception as e:
            log_fp.exception("process_file failed for %s", file_path)
            return self._err(str(e))

    def _pdf(self, fp: Path) -> Dict:
        doc = fitz.open(fp)
        meta = {
            "title":  doc.metadata.get("title") or fp.stem,
            "author": doc.metadata.get("author", "Unknown"),
            "pages":  doc.page_count,
        }
        blocks, full_text, images, img_count = [], [], [], 0

        for pn in range(doc.page_count):
            page = doc[pn]
            try:
                for blk in page.get_text("dict")["blocks"]:
                    if blk["type"] != 0:
                        continue
                    for line in blk.get("lines", []):
                        for span in line.get("spans", []):
                            t = span["text"].strip()
                            if t:
                                blocks.append({
                                    "type": "text", "content": t,
                                    "page": pn + 1, "size": span.get("size", 12),
                                })
                                full_text.append(t)
            except Exception as e:
                log_fp.warning("PDF page %d text extraction: %s", pn + 1, e)

            if img_count < MAX_IMAGES_PER_DOC:
                for img_info in page.get_images():
                    if img_count >= MAX_IMAGES_PER_DOC:
                        break
                    try:
                        bi  = doc.extract_image(img_info[0])
                        tb  = self._thumb(bi["image"], bi["ext"])
                        if tb:
                            images.append({
                                "page": pn + 1,
                                "data": f"data:image/{bi['ext']};base64,{tb}",
                            })
                        img_count += 1
                    except Exception as e:
                        log_fp.debug("PDF image extract p%d: %s", pn + 1, e)
        doc.close()
        return {
            "success":   True,
            "file_type": "pdf",
            "metadata":  meta,
            "content":   {
                "text":        " ".join(full_text),
                "blocks":      blocks,
                "image_count": img_count,
                "images":      images,
            },
            "html":  self._html(blocks, images, meta),
            "error": None,
        }

    def _docx(self, fp: Path) -> Dict:
        doc = docx.Document(fp)
        full_text, blocks = [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if t:
                blocks.append({
                    "type":    "paragraph",
                    "content": t,
                    "style":   p.style.name if p.style else "Normal",
                })
                full_text.append(t)
        for tbl in doc.tables:
            for row in tbl.rows:
                rt = " | ".join(c.text.strip() for c in row.cells if c.text.strip())
                if rt:
                    blocks.append({"type": "table_row", "content": rt})
                    full_text.append(rt)
        with open(fp, "rb") as f:
            html = mammoth.convert_to_html(f).value
        return {
            "success":   True,
            "file_type": "docx",
            "metadata":  {"title": fp.stem},
            "content":   {
                "text": "\n".join(full_text), "blocks": blocks,
                "image_count": 0, "images": [],
            },
            "html":  html,
            "error": None,
        }

    def _pptx(self, fp: Path) -> Dict:
        prs = Presentation(fp)
        blocks, full_text = [], []
        for sn, slide in enumerate(prs.slides, 1):
            for shape in slide.shapes:
                t = shape.text.strip() if hasattr(shape, "text") else ""
                if t:
                    blocks.append({"type": "slide_text", "content": t, "slide": sn})
                    full_text.append(t)
            if slide.has_notes_slide:
                nt = slide.notes_slide.notes_text_frame.text.strip()
                if nt:
                    blocks.append({"type": "slide_notes", "content": nt, "slide": sn})
                    full_text.append(nt)
        html = (
            f"<h1>{fp.stem}</h1>\n"
            + "".join(
                f'<p class="slide-{b["slide"]}">{b["content"]}</p>' for b in blocks
            )
        )
        return {
            "success":   True,
            "file_type": "pptx",
            "metadata":  {"title": fp.stem, "slides": len(prs.slides)},
            "content":   {
                "text": "\n".join(full_text), "blocks": blocks,
                "image_count": 0, "images": [],
            },
            "html":  html,
            "error": None,
        }

    def _text(self, fp: Path) -> Dict:
        content = open(fp, "r", encoding="utf-8", errors="replace").read()
        if fp.suffix == ".md":
            html = markdown.markdown(content, extensions=["tables", "fenced_code"])
        else:
            html = f"<pre>{content}</pre>"
        return {
            "success":   True,
            "file_type": "text",
            "metadata":  {"title": fp.stem},
            "content":   {
                "text": content,
                "blocks": [{"type": "text", "content": content}],
                "image_count": 0, "images": [],
            },
            "html":  html,
            "error": None,
        }

    @staticmethod
    def _thumb(raw: bytes, ext: str) -> Optional[str]:
        try:
            img = Image.open(BytesIO(raw))
            img.thumbnail(MAX_IMAGE_PIXELS, Image.LANCZOS)
            buf = BytesIO()
            fmt = "JPEG" if ext.lower() in ("jpg", "jpeg") else "PNG"
            img.convert("RGB").save(buf, format=fmt, quality=80)
            return base64.b64encode(buf.getvalue()).decode()
        except Exception as e:
            log_fp.debug("Thumbnail failed: %s", e)
            return None

    @staticmethod
    def _html(blocks: list, images: list, meta: dict) -> str:
        h = [f"<h1>{meta.get('title', 'Document')}</h1>"]
        for b in blocks:
            tag = "h2" if b.get("size", 12) >= 18 else "p"
            h.append(f"<{tag}>{b['content']}</{tag}>")
        for img in images:
            h.append(f'<img src="{img["data"]}" style="max-width:100%;margin:8px 0"/>')
        return "\n".join(h)

    @staticmethod
    def _err(msg: str) -> Dict:
        log_fp.error("FileProcessor error: %s", msg)
        return {
            "success":   False,
            "file_type": None,
            "metadata":  {},
            "content":   {"text": "", "blocks": [], "image_count": 0, "images": []},
            "html":      "",
            "error":     msg,
        }


file_processor = FileProcessor()
print(f"✅ Cell 5 complete — file processor ready ({len(file_processor.SUPPORTED_FORMATS)} formats)")

✅ Cell 5 complete — file processor ready (5 formats)


# 6. LLM Engine (Smart Teaching System)

In [13]:
import asyncio, threading
from collections import deque, OrderedDict
import torch

log_llm = logging.getLogger("llm")


class LRUCache:
    def __init__(self, maxsize: int = 512):
        self._d   = OrderedDict()
        self._max = maxsize
        self._lock = threading.Lock()

    def get(self, k):
        with self._lock:
            if k not in self._d:
                return None
            self._d.move_to_end(k)
            return self._d[k]

    def set(self, k, v):
        with self._lock:
            if k in self._d:
                self._d.move_to_end(k)
            self._d[k] = v
            if len(self._d) > self._max:
                self._d.popitem(last=False)

    def __contains__(self, k):
        return self.get(k) is not None


class SmartTeachingEngine:
    MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"
    MAX_SEQ_LEN   = 4096
    CONTEXT_LIMIT = 2048

    SYSTEM_PROMPTS = {
        "general":               "You are a helpful AI teaching assistant. Be clear, accurate, and encouraging.",
        "quiz_generator":        (
            "You are a quiz generator. Create high-quality multiple-choice questions. "
            "Respond ONLY with a JSON array. Each element must have: "
            "question (str), options (list[str] of 4), correct (int 0-3), explanation (str)."
        ),
        "explainer":             "You are an expert teacher. Explain concepts clearly using examples and analogies.",
        "summarizer":            "You are a summarisation expert. Be concise and preserve key points.",
        "qa_assistant":          (
            "You are a Q&A assistant. Answer ONLY using the provided context. "
            "If the context does not contain the answer, say so explicitly."
        ),
        "socratic":              (
            "You are a Socratic tutor. Guide the student to the answer through questions "
            "rather than stating the answer directly."
        ),
        "misconception_detector": "You are a learning diagnostician. Identify and gently correct misconceptions.",
        "study_planner":          "You are a study plan architect. Create structured, achievable study plans.",
    }

    def __init__(self):
        self.device       = "cuda" if torch.cuda.is_available() else "cpu"
        self.model        = None
        self.tokenizer    = None
        self._loaded      = False
        self._load_lock   = threading.Lock()
        self._load_error  = None
        self._memory      = LRUCache(256)
        self._lecture_ctx = LRUCache(128)
        self._profiles    = LRUCache(512)

    @property
    def is_loaded(self) -> bool:
        return self._loaded

    @property
    def load_error(self) -> Optional[str]:
        return self._load_error

    def load(self, use_4bit: bool = True) -> bool:
        with self._load_lock:
            if self._loaded:
                return True
            return self._do_load(use_4bit)

    def _do_load(self, use_4bit: bool) -> bool:
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        log_llm.info("Loading %s (4bit=%s, device=%s)", self.MODEL_NAME, use_4bit, self.device)
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.MODEL_NAME, trust_remote_code=True
            )
            if not self.tokenizer.pad_token:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            kw: Dict[str, Any] = {"trust_remote_code": True, "device_map": "auto"}
            if use_4bit and self.device == "cuda":
                kw["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                )
            else:
                kw["torch_dtype"] = torch.float16 if self.device == "cuda" else torch.float32

            self.model = AutoModelForCausalLM.from_pretrained(self.MODEL_NAME, **kw)
            self.model.eval()
            self._loaded     = True
            self._load_error = None
            log_llm.info("Model loaded ✅  (device=%s)", self.device)
            return True
        except Exception as e:
            log_llm.error("Model load failed: %s", e)
            self._load_error = str(e)
            if use_4bit:
                log_llm.info("Retrying without 4-bit quantisation…")
                return self._do_load(False)
            return False

    def load_async(self) -> threading.Thread:
        t = threading.Thread(target=self.load, daemon=True, name="llm-loader")
        t.start()
        return t

    def wait_until_loaded(self, timeout: float = 600.0) -> bool:
        deadline = time.monotonic() + timeout
        while not self._loaded and not self._load_error:
            if time.monotonic() > deadline:
                log_llm.error("LLM load timed out after %.0fs", timeout)
                return False
            time.sleep(1.0)
        return self._loaded

    # ── Student profiles ──────────────────────────────────────

    def get_student_profile(self, user_id: str) -> Dict:
        p = self._profiles.get(user_id)
        if p is None:
            p = {
                "level":          "beginner",
                "weak_topics":    [],
                "strong_topics":  [],
                "learning_style": "visual",
                "quiz_history":   [],
                "misconceptions": [],
                "preferences":    {"detail_level": "medium", "examples": True, "analogies": True},
            }
            self._profiles.set(user_id, p)
        return p

    def update_student_profile(self, user_id: str, **kw):
        p = self.get_student_profile(user_id)
        p.update(kw)
        self._profiles.set(user_id, p)

    # ── Context helpers ───────────────────────────────────────

    def _level_prefix(self, user_id: Optional[str]) -> str:
        if not user_id:
            return ""
        level = self.get_student_profile(user_id).get("level", "beginner")
        return {
            "beginner":     "Explain simply, no prior knowledge assumed: ",
            "intermediate": "Explain clearly with depth: ",
            "advanced":     "Provide in-depth technical explanation: ",
        }.get(level, "")

    def set_lecture_context(self, session_id: str, content: str):
        self._lecture_ctx.set(session_id, self._truncate(content, self.CONTEXT_LIMIT))

    def append_lecture_context(self, session_id: str, new_text: str):
        cur = self._lecture_ctx.get(session_id) or ""
        self.set_lecture_context(session_id, f"{cur}\n{new_text}".strip())

    def add_to_memory(self, session_id: str, role: str, content: str):
        mem = self._memory.get(session_id)
        if mem is None:
            mem = deque(maxlen=12)
        mem.append({"role": role, "content": content})
        self._memory.set(session_id, mem)

    def get_memory(self, session_id: Optional[str]) -> List[Dict]:
        if not session_id:
            return []
        mem = self._memory.get(session_id)
        return list(mem) if mem else []

    def _build_messages(
        self, user_msgs: List[Dict], mode: str,
        session_id: Optional[str], user_id: Optional[str]
    ) -> List[Dict]:
        msgs = [{"role": "system", "content": self.SYSTEM_PROMPTS.get(mode, self.SYSTEM_PROMPTS["general"])}]
        ctx = self._lecture_ctx.get(session_id) if session_id else None
        if ctx:
            msgs.append({"role": "system", "content": f"Lecture material:\n{ctx}"})
        msgs.extend(self.get_memory(session_id)[-6:])
        adapted = [dict(m) for m in user_msgs]
        prefix  = self._level_prefix(user_id)
        if prefix and adapted and adapted[-1]["role"] == "user":
            adapted[-1]["content"] = prefix + adapted[-1]["content"]
        msgs.extend(adapted)
        return msgs

    def _tokenise(self, messages: List[Dict], reserve: int):
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return self.tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=self.MAX_SEQ_LEN - reserve,
        ).to(self.device)

    def _truncate(self, text: str, max_tok: int) -> str:
        if not self.tokenizer:
            return text[:max_tok * 4]
        toks = self.tokenizer.encode(text, add_special_tokens=False)
        if len(toks) <= max_tok:
            return text
        h = max_tok // 2
        return self.tokenizer.decode(toks[:h] + toks[-h:], skip_special_tokens=True)

    def chat(
        self, messages: List[Dict], mode: str = "general",
        max_tokens: int = 512, temperature: float = 0.7,
        user_id: Optional[str] = None, session_id: Optional[str] = None
    ) -> str:
        if not self._loaded:
            log_llm.error("chat() called before model loaded")
            return "[ERROR: Model not loaded yet]"
        try:
            inputs = self._tokenise(
                self._build_messages(messages, mode, session_id, user_id), max_tokens
            )
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens     = max_tokens,
                    temperature        = max(temperature, 0.01),
                    do_sample          = True,
                    top_p              = 0.9,
                    repetition_penalty = 1.1,
                    pad_token_id       = self.tokenizer.pad_token_id,
                    eos_token_id       = self.tokenizer.eos_token_id,
                )
            resp = self.tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            ).strip()
            if session_id:
                self.add_to_memory(session_id, "assistant", resp)
            return resp
        except Exception as e:
            log_llm.exception("chat() failed (mode=%s)", mode)
            return f"[ERROR: {e}]"

    def stream_chat(
        self, messages: List[Dict], mode: str = "general",
        max_tokens: int = 512, temperature: float = 0.7,
        user_id: Optional[str] = None, session_id: Optional[str] = None
    ):
        from transformers import TextIteratorStreamer
        if not self._loaded:
            log_llm.error("stream_chat() called before model loaded")
            yield "[ERROR: Model not loaded yet]"
            return
        try:
            inputs   = self._tokenise(
                self._build_messages(messages, mode, session_id, user_id), max_tokens
            )
            streamer = TextIteratorStreamer(
                self.tokenizer, skip_prompt=True, skip_special_tokens=True
            )
            gen_kwargs = {
                **inputs,
                "streamer":          streamer,
                "max_new_tokens":    max_tokens,
                "temperature":       max(temperature, 0.01),
                "do_sample":         True,
                "top_p":             0.9,
                "repetition_penalty": 1.1,
                "pad_token_id":      self.tokenizer.pad_token_id,
                "eos_token_id":      self.tokenizer.eos_token_id,
            }
            t    = threading.Thread(target=self.model.generate, kwargs=gen_kwargs, daemon=True)
            t.start()
            full = ""
            try:
                for chunk in streamer:
                    full += chunk
                    yield chunk
            finally:
                t.join(timeout=60)
            if session_id:
                self.add_to_memory(session_id, "assistant", full)
        except Exception as e:
            log_llm.exception("stream_chat() failed")
            yield f"[ERROR: {e}]"

    def summarize(self, content: str, max_words: int = 200) -> str:
        return self.chat(
            [{"role": "user", "content": f"Summarise in ≈{max_words} words:\n\n{content[:6000]}"}],
            mode="summarizer",
        )

    def generate_quiz(
        self, content: str, num_questions: int = 5, difficulty: str = "medium"
    ) -> Optional[list]:
        import json, re
        prompt = (
            f"Generate exactly {num_questions} {difficulty}-difficulty MCQs from:\n\n{content[:4000]}"
        )
        raw = self.chat([{"role": "user", "content": prompt}], mode="quiz_generator", max_tokens=1024)
        try:
            m = re.search(r"\[.*\]", raw, re.DOTALL)
            if m:
                return json.loads(m.group())
        except Exception as e:
            log_llm.warning("Quiz JSON parse failed: %s — raw: %.200s", e, raw)
        return None


teaching_engine = SmartTeachingEngine()
log_llm.info("LLM loading in background thread…")
_llm_thread = teaching_engine.load_async()
print("✅ Cell 6 complete — LLM initialised (loading async)")

✅ Cell 6 complete — LLM initialised (loading async)


# 7. RAG Engine (Retrieval-Augmented Generation)

In [14]:
import asyncio
import numpy as np
from sentence_transformers import SentenceTransformer

log_rag = logging.getLogger("rag")


class EmbeddingEngine:
    MODEL_NAME = "all-MiniLM-L6-v2"

    def __init__(self):
        dev        = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(self.MODEL_NAME, device=dev)
        self.dim   = self.model.get_sentence_embedding_dimension()
        log_rag.info("Embedder ready dim=%d device=%s", self.dim, dev)

    def encode(self, text: str) -> List[float]:
        return self.model.encode(text, convert_to_tensor=False).tolist()

    def encode_batch(self, texts: List[str], batch_size: int = 64) -> List[List[float]]:
        return self.model.encode(
            texts, batch_size=batch_size,
            convert_to_tensor=False, show_progress_bar=False,
        ).tolist()


class DocumentChunker:
    def __init__(self, chunk_size: int = 512, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap    = overlap

    def chunk(self, text: str) -> List[Dict]:
        sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
        chunks, current, cw = [], [], 0

        for i, sent in enumerate(sentences):
            sw = len(sent.split())
            if cw + sw > self.chunk_size and current:
                chunks.append({
                    "text":       ". ".join(current) + ".",
                    "start_sent": i - len(current),
                    "end_sent":   i,
                })
                kept, kw = [], 0
                for s in reversed(current):
                    sw2 = len(s.split())
                    if kw + sw2 > self.overlap:
                        break
                    kept.insert(0, s)
                    kw += sw2
                current, cw = kept + [sent], kw + sw
            else:
                current.append(sent)
                cw += sw

        if current:
            chunks.append({
                "text":       ". ".join(current) + ".",
                "start_sent": len(sentences) - len(current),
                "end_sent":   len(sentences),
            })
        return chunks


class RAGEngine:
    def __init__(self, llm, embedder, db, cache):
        self.llm      = llm
        self.embedder = embedder
        self.db       = db
        self.cache    = cache
        self.chunker  = DocumentChunker(512, 50)

    async def ingest_document(
        self, lecture_id: str, content: str, metadata: dict = None
    ) -> Dict:
        t0     = time.monotonic()
        chunks = self.chunker.chunk(content)
        if not chunks:
            log_rag.warning("No chunks produced for lecture %s", lecture_id)
            return {"lecture_id": lecture_id, "chunks": 0, "stored": 0, "elapsed_s": 0.0}

        embeddings = await asyncio.to_thread(
            self.embedder.encode_batch, [c["text"] for c in chunks]
        )
        now  = datetime.utcnow().isoformat()
        rows = [
            {
                "lecture_id":   lecture_id,
                "chunk_index":  i,
                "content":      c["text"],
                "embedding":    e,
                "metadata":     {
                    **(metadata or {}),
                    "word_count": len(c["text"].split()),
                    "start_sent": c["start_sent"],
                    "end_sent":   c["end_sent"],
                },
                "created_at":   now,
            }
            for i, (c, e) in enumerate(zip(chunks, embeddings))
        ]
        stored  = await asyncio.to_thread(self.db.store_embeddings_batch, rows)
        elapsed = time.monotonic() - t0
        log_rag.info(
            "Ingested %s: %d chunks, %d stored, %.1fs",
            lecture_id, len(chunks), stored, elapsed
        )
        return {
            "lecture_id": lecture_id,
            "chunks":     len(chunks),
            "stored":     stored,
            "elapsed_s":  round(elapsed, 2),
        }

    async def search(
        self, query: str, lecture_id: Optional[str] = None, top_k: int = 5
    ) -> List[Dict]:
        qemb    = await asyncio.to_thread(self.embedder.encode, query)
        results = await asyncio.to_thread(self.db.search_similar, qemb, lecture_id, top_k)
        for r in results:
            r["similarity"] = float(np.clip(r.get("similarity", 0.0), 0.0, 1.0))
        return results

    async def answer_question(
        self, query: str, lecture_id: Optional[str] = None,
        user_id: Optional[str] = None, session_id: Optional[str] = None,
        top_k: int = 5,
    ) -> Dict:
        results = await self.search(query, lecture_id, top_k)
        if not results:
            return {"answer": "No relevant information found.", "sources": [], "confidence": 0.0}

        ctx    = "\n\n".join(f"[Source {i+1}] {d['content']}" for i, d in enumerate(results))
        answer = await asyncio.to_thread(
            self.llm.chat,
            [{"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {query}"}],
            "qa_assistant", 512, 0.3, user_id, session_id,
        )
        conf = float(np.mean([r.get("similarity", 0.0) for r in results]))
        return {
            "answer":     answer,
            "sources":    results,
            "confidence": round(conf, 3),
        }

    async def stream_answer(
        self, query: str, lecture_id: Optional[str] = None,
        user_id: Optional[str] = None, session_id: Optional[str] = None,
        top_k: int = 5,
    ):
        results = await self.search(query, lecture_id, top_k)
        if not results:
            yield "No relevant context found."
            return
        ctx = "\n\n".join(f"[Source {i+1}] {d['content']}" for i, d in enumerate(results))
        for chunk in self.llm.stream_chat(
            [{"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {query}"}],
            mode="qa_assistant", temperature=0.3,
            user_id=user_id, session_id=session_id,
        ):
            yield chunk

    async def add_live_text(self, session_id: str, text: str):
        self.llm.append_lecture_context(session_id, text)


log_rag.info("Loading embedding model…")
embedder   = EmbeddingEngine()
rag_engine = RAGEngine(teaching_engine, embedder, supabase_db, local_cache)
print(f"✅ Cell 7 complete — RAG engine ready (dim={embedder.dim})")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Cell 7 complete — RAG engine ready (dim=384)


# 8. Live Transcription Engine

In [15]:
import asyncio, hashlib, re, threading
from collections import deque
from dataclasses import dataclass, field
import numpy as np, webrtcvad

log_stt    = logging.getLogger("stt")
SAMPLE_RATE     = 16000
CHUNK_DURATION  = 5.0
BUFFER_SECONDS  = 30
MIN_TEXT_LEN    = 5


@dataclass
class SessionState:
    session_id:       str
    title:            str
    started_at:       float = field(default_factory=time.time)
    audio_buffer:     deque = field(default_factory=lambda: deque(maxlen=SAMPLE_RATE * BUFFER_SECONDS))
    last_process_time: float = 0.0
    prev_hash:        Optional[str] = None
    total_transcripts: int = 0
    chunks_processed: int  = 0


class LiveTranscriptionEngine:
    def __init__(self, model_size: str = "base"):
        self.model_size = model_size
        self.device     = "cuda" if torch.cuda.is_available() else "cpu"
        self.model      = None
        self.loaded     = False
        self.vad        = webrtcvad.Vad(2)
        self._sessions: Dict[str, SessionState] = {}
        self._lock      = threading.Lock()

    def load(self) -> bool:
        if self.loaded:
            return True
        try:
            from faster_whisper import WhisperModel
            self.model  = WhisperModel(
                self.model_size, device=self.device,
                compute_type="float16" if self.device == "cuda" else "int8",
            )
            self.loaded = True
            log_stt.info("Whisper '%s' loaded ✅ device=%s", self.model_size, self.device)
            return True
        except Exception as e:
            log_stt.error("Whisper load failed: %s", e)
            return False

    def start_session(self, session_id: str, title: str = "Live Lecture") -> str:
        with self._lock:
            self._sessions[session_id] = SessionState(session_id=session_id, title=title)
        local_cache.create_session(session_id, title)
        log_stt.info("STT session started: %s '%s'", session_id, title)
        return session_id

    def end_session(self, session_id: str) -> Dict:
        with self._lock:
            state = self._sessions.pop(session_id, None)
        if not state:
            log_stt.warning("end_session: session %s not found", session_id)
            return {"error": f"Session '{session_id}' not found"}

        local_cache.end_session(session_id)     # FIX #7 — method now exists
        result = {
            "session_id":        session_id,
            "title":             state.title,
            "duration_seconds":  round(time.time() - state.started_at, 1),
            "total_transcripts": state.total_transcripts,
            "chunks_processed":  state.chunks_processed,
        }
        log_stt.info("STT session ended: %s — %s", session_id, result)
        return result

    def get_active_sessions(self) -> List[str]:
        with self._lock:
            return list(self._sessions.keys())

    async def process_audio_chunk(
        self, audio_bytes: bytes, session_id: str
    ) -> Optional[Dict]:
        if not self.loaded:
            log_stt.warning("process_audio_chunk: model not loaded")
            return None
        with self._lock:
            state = self._sessions.get(session_id)
        if not state:
            log_stt.debug("process_audio_chunk: unknown session %s", session_id)
            return None

        audio = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
        state.audio_buffer.extend(audio.tolist())

        now = time.time()
        if now - state.last_process_time < CHUNK_DURATION:
            return None
        state.last_process_time = now

        cs = int(SAMPLE_RATE * 4.0)
        if len(state.audio_buffer) < cs:
            return None

        chunk = np.array(list(state.audio_buffer)[-cs:], dtype=np.float32)
        if not self._vad(chunk):
            return None

        try:
            # FIX #9: wrap in list() so the generator is fully consumed before joining
            segs, info = await asyncio.to_thread(
                self.model.transcribe, chunk, language="en", vad_filter=True, beam_size=5
            )
            segs = list(segs)   # ← materialise generator before thread exits
        except Exception as e:
            log_stt.error("Transcription failed for session %s: %s", session_id, e)
            return None

        text = self._clean(" ".join(s.text.strip() for s in segs))
        if not text or len(text) < MIN_TEXT_LEN:
            return None

        h = hashlib.md5(text.encode()).hexdigest()
        if h == state.prev_hash:
            return None
        state.prev_hash = h

        speaker = self._speaker(chunk)
        result  = {
            "session_id": session_id,
            "speaker":    speaker,
            "text":       text,
            "timestamp":  now,
            "confidence": float(getattr(info, "language_probability", 0.0)),
        }
        local_cache.cache_transcript(session_id, speaker, text, now)
        state.total_transcripts += 1
        state.chunks_processed  += 1
        return result

    def _vad(self, audio: np.ndarray) -> bool:
        try:
            pcm = (audio * 32768).astype(np.int16).tobytes()
            fs  = int(SAMPLE_RATE * 0.03) * 2
            for i in range(0, len(pcm), fs):
                f = pcm[i : i + fs]
                if len(f) == fs and self.vad.is_speech(f, SAMPLE_RATE):
                    return True
            return False
        except Exception:
            return True     # fail-open

    @staticmethod
    def _speaker(audio: np.ndarray) -> str:
        e = float(np.sqrt(np.mean(audio ** 2)))
        return "Teacher" if e > 0.05 else ("Student" if e > 0.02 else "Unknown")

    @staticmethod
    def _clean(text: str) -> str:
        text = re.sub(r"\s+", " ", text.strip())
        if text and text[-1] not in ".!?":
            text += "."
        return text

    def get_stats(self) -> Dict:
        with self._lock:
            return {
                "loaded":          self.loaded,
                "active_sessions": list(self._sessions.keys()),
            }


async def transcription_sync_worker():
    """
    FIX #8: This coroutine is now scheduled inside the FastAPI lifespan
    (Cell 9) where the event loop is guaranteed to be running.
    """
    log_stt.info("Transcript sync worker started")
    while True:
        try:
            rows = local_cache.pop_unsynced_transcripts(200)
            if rows:
                records = [
                    {
                        "session_id": r["session_id"],
                        "speaker":    r["speaker"],
                        "transcript": r["text"],
                        "timestamp":  r["timestamp"],
                    }
                    for r in rows
                ]
                ok = await asyncio.to_thread(supabase_db.bulk_insert_transcripts, records)
                log_stt.info("Synced %d transcripts (ok=%s)", len(rows), ok)
        except Exception as e:
            log_stt.error("Transcript sync error: %s", e)
        await asyncio.sleep(10)


import nest_asyncio
nest_asyncio.apply()

stt_engine = LiveTranscriptionEngine(model_size="base")
stt_engine.load()
# NOTE: sync worker is started in the FastAPI lifespan below (Cell 9)
print(f"✅ Cell 8 complete — STT ready (Whisper {stt_engine.model_size})")

✅ Cell 8 complete — STT ready (Whisper base)


# 9. FastAPI + Socket.IO Server

In [16]:
import asyncio, secrets as sec_mod
from contextlib import asynccontextmanager

import aiofiles
import socketio
from fastapi import (
    BackgroundTasks, Depends, FastAPI, File, Header,
    HTTPException, Query, UploadFile, status,
)
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field

log_api = logging.getLogger("api")

# ── Pydantic schemas ──────────────────────────────────────────

class RegisterRequest(BaseModel):
    username:       str = Field(..., min_length=3, max_length=50)
    email:          str = Field(..., min_length=5)
    password:       str = Field(..., min_length=8)
    role:           str = Field("student")
    learning_style: str = Field("visual")

class LoginRequest(BaseModel):
    username: str
    password: str

class ChatRequest(BaseModel):
    message:    str
    mode:       str = "general"
    session_id: Optional[str] = None
    lecture_id: Optional[str] = None
    max_tokens: int = Field(512, ge=64, le=2048)
    temperature: float = Field(0.7, ge=0.0, le=1.0)

class QuizRequest(BaseModel):
    lecture_id:     str
    num_questions:  int = Field(5, ge=1, le=20)
    difficulty:     str = "medium"

class QuizSubmitRequest(BaseModel):
    lecture_id: Optional[str] = None
    questions:  list
    answers:    list
    difficulty: str = "medium"

class SessionCreateRequest(BaseModel):
    title:     str = "Live Lecture"
    course_id: Optional[str] = None

# ── Socket.IO ─────────────────────────────────────────────────

sio = socketio.AsyncServer(
    async_mode="asgi",
    cors_allowed_origins="*",
    logger=False,
    engineio_logger=False,
)

# ── Session manager ───────────────────────────────────────────

class SessionManager:
    def __init__(self):
        self._sessions: Dict[str, Dict] = {}
        self._lock      = asyncio.Lock()

    async def create(self, title: str, teacher_id: Optional[str] = None) -> str:
        sid = str(uuid.uuid4())
        async with self._lock:
            self._sessions[sid] = {
                "id":         sid,
                "title":      title,
                "teacher_id": teacher_id,
                "created_at": time.time(),
                "transcript": deque(maxlen=500),
                "user_sids":  set(),
            }
        return sid

    async def get(self, session_id: str) -> Optional[Dict]:
        async with self._lock:
            s = self._sessions.get(session_id)
            if s is None:
                return None
            return {**s, "transcript": list(s["transcript"]), "user_sids": list(s["user_sids"])}

    async def list_all(self) -> List[Dict]:
        async with self._lock:
            return [
                {k: v for k, v in s.items() if k not in ("transcript", "user_sids")}
                for s in self._sessions.values()
            ]

    async def add_transcript(self, session_id: str, entry: Dict):
        async with self._lock:
            s = self._sessions.get(session_id)
            if s:
                s["transcript"].append(entry)

    async def add_user_sid(self, session_id: str, socket_sid: str):
        async with self._lock:
            s = self._sessions.get(session_id)
            if s:
                s["user_sids"].add(socket_sid)

    async def remove_user_sid(self, socket_sid: str):
        async with self._lock:
            for s in self._sessions.values():
                s["user_sids"].discard(socket_sid)

    async def delete(self, session_id: str) -> Optional[Dict]:
        async with self._lock:
            return self._sessions.pop(session_id, None)


session_manager = SessionManager()

# ── Auth dependencies ─────────────────────────────────────────

async def get_current_user(authorization: Optional[str] = Header(None)) -> Dict:
    """Extract and validate Bearer JWT from Authorization header."""
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Missing or malformed Authorization header.",
        )
    token  = authorization.split(" ", 1)[1]
    result = auth_manager.verify_token(token)
    if not result.ok:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail=result.error or "Invalid token.",
        )
    return result.payload


def require_role(*roles: str):
    """Dependency factory: allow only specific roles."""
    async def _dep(user: Dict = Depends(get_current_user)) -> Dict:
        if user.get("role") not in roles:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Role '{user.get('role')}' is not permitted. Required: {list(roles)}",
            )
        return user
    return _dep


# ── Lifespan ──────────────────────────────────────────────────

@asynccontextmanager
async def lifespan(application: FastAPI):
    """
    FIX #8: Sync worker is created here, inside the running event loop.
    """
    log_api.info("Server starting up…")
    sync_task = asyncio.create_task(transcription_sync_worker())
    yield
    log_api.info("Server shutting down…")
    sync_task.cancel()
    try:
        await sync_task
    except asyncio.CancelledError:
        pass


# ── FastAPI app ───────────────────────────────────────────────

app = FastAPI(
    title="Smart Classroom AI",
    version="1.0.0",
    description="AI-powered classroom assistant with RAG, live STT, and adaptive quizzing.",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

socket_app = socketio.ASGIApp(sio, app)

# ── Health ────────────────────────────────────────────────────

@app.get("/health", tags=["system"])
async def health():
    return {
        "status":    "ok",
        "llm":       teaching_engine.is_loaded,
        "stt":       stt_engine.loaded,
        "db":        supabase_db.connected,
        "formats":   list(file_processor.SUPPORTED_FORMATS.keys()),
        "timestamp": datetime.utcnow().isoformat(),
    }


# ── Auth endpoints ────────────────────────────────────────────

@app.post("/auth/register", status_code=201, tags=["auth"])
async def register(body: RegisterRequest):
    uid, err = auth_manager.register_user(
        body.username, body.email, body.password, body.role, body.learning_style
    )
    if err:
        raise HTTPException(status_code=400, detail=err)
    return {"user_id": uid, "username": body.username, "role": body.role}


@app.post("/auth/login", tags=["auth"])
async def login(body: LoginRequest):
    user, err = auth_manager.authenticate(body.username, body.password)
    if err:
        raise HTTPException(status_code=401, detail=err)
    token = auth_manager.generate_token(user)
    return {"access_token": token, "token_type": "bearer", "user": user}


@app.get("/auth/me", tags=["auth"])
async def me(user: Dict = Depends(get_current_user)):
    return user


# ── Lecture endpoints ─────────────────────────────────────────

@app.post("/lectures/upload", status_code=201, tags=["lectures"])
async def upload_lecture(
    file:      UploadFile = File(...),
    title:     Optional[str] = Query(None),
    course_id: Optional[str] = Query(None),
    user:      Dict = Depends(require_role("teacher", "admin")),
):
    suffix = Path(file.filename).suffix.lower()
    if suffix not in file_processor.SUPPORTED_FORMATS:
        raise HTTPException(
            status_code=415,
            detail=f"Unsupported format '{suffix}'. Accepted: {list(file_processor.SUPPORTED_FORMATS.keys())}",
        )

    temp_id = sec_mod.token_hex(8)
    dest    = DIRS["uploads"] / f"{temp_id}{suffix}"

    # FIX #22: use aiofiles for non-blocking file writes
    async with aiofiles.open(dest, "wb") as fh:
        while chunk := await file.read(1 << 20):
            await fh.write(chunk)

    proc = file_processor.process_file(dest)
    if not proc["success"]:
        dest.unlink(missing_ok=True)
        raise HTTPException(status_code=422, detail=proc.get("error", "Processing failed"))

    lid = supabase_db.create_lecture(
        title or file.filename,
        course_id,
        user["user_id"],
        proc["file_type"],
        str(dest),
        proc["metadata"],
    )
    if not lid:
        raise HTTPException(status_code=500, detail="Failed to register lecture in database.")

    rag = await rag_engine.ingest_document(lid, proc["content"]["text"], proc["metadata"])
    log_api.info("Lecture uploaded: %s → %s (%d chunks)", file.filename, lid, rag["chunks"])
    return {"lecture_id": lid, "title": title or file.filename, "rag": rag}


@app.get("/lectures", tags=["lectures"])
async def list_lectures(
    course_id: Optional[str] = Query(None),
    limit:     int = Query(50, ge=1, le=200),
    _user:     Dict = Depends(get_current_user),
):
    return supabase_db.get_lectures(course_id=course_id, limit=limit)


# ── Chat / RAG endpoints ──────────────────────────────────────

@app.post("/ask", tags=["ai"])
async def ask(body: ChatRequest, user: Dict = Depends(get_current_user)):
    if not teaching_engine.is_loaded:
        raise HTTPException(status_code=503, detail="LLM not ready yet — please wait.")

    uid = user["user_id"]
    if body.lecture_id:
        result = await rag_engine.answer_question(
            body.message, body.lecture_id, uid, body.session_id
        )
        return result

    answer = await asyncio.to_thread(
        teaching_engine.chat,
        [{"role": "user", "content": body.message}],
        body.mode, body.max_tokens, body.temperature, uid, body.session_id,
    )
    return {"answer": answer, "sources": [], "confidence": None}


@app.post("/ask/stream", tags=["ai"])
async def ask_stream(body: ChatRequest, user: Dict = Depends(get_current_user)):
    if not teaching_engine.is_loaded:
        raise HTTPException(status_code=503, detail="LLM not ready yet.")

    uid = user["user_id"]

    async def generator():
        if body.lecture_id:
            async for chunk in rag_engine.stream_answer(
                body.message, body.lecture_id, uid, body.session_id
            ):
                yield chunk
        else:
            for chunk in teaching_engine.stream_chat(
                [{"role": "user", "content": body.message}],
                body.mode, body.max_tokens, body.temperature, uid, body.session_id,
            ):
                yield chunk

    return StreamingResponse(generator(), media_type="text/plain")


# ── Quiz endpoints ────────────────────────────────────────────

@app.post("/quiz/generate", tags=["quiz"])
async def generate_quiz(
    body: QuizRequest,
    user: Dict = Depends(get_current_user),
):
    if not teaching_engine.is_loaded:
        raise HTTPException(status_code=503, detail="LLM not ready.")

    results = await rag_engine.search(
        f"key concepts from lecture {body.lecture_id}", body.lecture_id, top_k=10
    )
    if not results:
        raise HTTPException(status_code=404, detail="No content found for this lecture.")

    content = " ".join(r["content"] for r in results)
    quiz    = await asyncio.to_thread(
        teaching_engine.generate_quiz, content, body.num_questions, body.difficulty
    )
    if not quiz:
        raise HTTPException(status_code=500, detail="Quiz generation failed — try again.")

    return {"questions": quiz, "lecture_id": body.lecture_id, "difficulty": body.difficulty}


@app.post("/quiz/submit", tags=["quiz"])
async def submit_quiz(
    body: QuizSubmitRequest,
    user: Dict = Depends(get_current_user),
):
    if len(body.questions) != len(body.answers):
        raise HTTPException(status_code=400, detail="questions/answers length mismatch.")

    correct = sum(
        1 for q, a in zip(body.questions, body.answers)
        if isinstance(q, dict) and q.get("correct") == a
    )
    score = (correct / len(body.questions)) * 100 if body.questions else 0.0

    supabase_db.save_quiz_result(
        user["user_id"], body.lecture_id,
        body.questions, body.answers, score, body.difficulty,
    )
    # Update in-memory profile
    teaching_engine.update_student_profile(
        user["user_id"],
        quiz_history=[{"score": score, "difficulty": body.difficulty, "ts": datetime.utcnow().isoformat()}],
    )
    return {"score": round(score, 1), "correct": correct, "total": len(body.questions)}


# ── Live session endpoints ────────────────────────────────────

@app.post("/sessions", status_code=201, tags=["sessions"])
async def create_session(
    body: SessionCreateRequest,
    user: Dict = Depends(require_role("teacher", "admin")),
):
    sid = await session_manager.create(body.title, teacher_id=user["user_id"])
    stt_engine.start_session(sid, body.title)
    supabase_db.create_session_record(sid, body.title, user["user_id"], body.course_id)
    log_api.info("Session created: %s '%s'", sid, body.title)
    return {"session_id": sid, "title": body.title}


@app.get("/sessions", tags=["sessions"])
async def list_sessions(_user: Dict = Depends(get_current_user)):
    return await session_manager.list_all()


@app.get("/sessions/{session_id}", tags=["sessions"])
async def get_session(session_id: str, _user: Dict = Depends(get_current_user)):
    s = await session_manager.get(session_id)
    if not s:
        raise HTTPException(status_code=404, detail="Session not found.")
    return s


@app.delete("/sessions/{session_id}", tags=["sessions"])
async def end_session(
    session_id: str,
    user: Dict = Depends(require_role("teacher", "admin")),
):
    s = await session_manager.delete(session_id)
    if not s:
        raise HTTPException(status_code=404, detail="Session not found.")
    stats = stt_engine.end_session(session_id)
    supabase_db.end_session_record(session_id)
    log_api.info("Session ended: %s", session_id)
    return {"ended": True, "stats": stats}


# ── Socket.IO events ──────────────────────────────────────────

@sio.event
async def connect(sid, environ, auth):
    token = (auth or {}).get("token", "")
    result = auth_manager.verify_token(token)
    if not result.ok:
        log_api.warning("Socket rejected (invalid token): %s", sid)
        raise ConnectionRefusedError("Authentication failed.")
    log_api.info("Socket connected: %s (%s)", sid, result.payload.get("username"))

@sio.event
async def disconnect(sid):
    await session_manager.remove_user_sid(sid)
    log_api.info("Socket disconnected: %s", sid)

@sio.event
async def join_session(sid, data):
    session_id = data.get("session_id")
    if not session_id:
        return {"error": "session_id required"}
    await sio.enter_room(sid, session_id)
    await session_manager.add_user_sid(session_id, sid)
    log_api.info("Socket %s joined session %s", sid, session_id)
    return {"joined": session_id}

@sio.event
async def audio_chunk(sid, data):
    session_id  = data.get("session_id")
    audio_bytes = data.get("audio")
    if not session_id or not audio_bytes:
        return
    if isinstance(audio_bytes, str):
        import base64 as b64
        audio_bytes = b64.b64decode(audio_bytes)
    result = await stt_engine.process_audio_chunk(audio_bytes, session_id)
    if result:
        await session_manager.add_transcript(session_id, result)
        await rag_engine.add_live_text(session_id, result["text"])
        await sio.emit("transcript", result, room=session_id)

@sio.event
async def chat_message(sid, data):
    session_id = data.get("session_id")
    message    = data.get("message", "")
    user_id    = data.get("user_id")
    if not message:
        return
    if not teaching_engine.is_loaded:
        await sio.emit("chat_response", {"error": "LLM not ready"}, to=sid)
        return
    answer = await asyncio.to_thread(
        teaching_engine.chat,
        [{"role": "user", "content": message}],
        "general", 512, 0.7, user_id, session_id,
    )
    await sio.emit("chat_response", {"answer": answer, "session_id": session_id}, to=sid)

print("✅ Cell 9 complete — FastAPI + Socket.IO server defined")

✅ Cell 9 complete — FastAPI + Socket.IO server defined


# 10. Server Deployment

In [ ]:
import socket, subprocess, threading, asyncio, logging
from collections import deque

log_dep = logging.getLogger("deploy")
PORT    = 8000


def free_port(port: int, max_wait: float = 10.0):
    subprocess.run(
        f"fuser -k {port}/tcp 2>/dev/null || true",
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    deadline = time.monotonic() + max_wait
    while time.monotonic() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", port))
                return
            except OSError:
                time.sleep(0.5)
    raise RuntimeError(f"Port {port} still in use after {max_wait}s")


def start_ngrok(port: int) -> Optional[str]:
    from pyngrok import ngrok
    token = os.environ.get("NGROK_AUTH_TOKEN")
    if not token:
        log_dep.warning("NGROK_AUTH_TOKEN not set — skipping tunnel")
        return None
    try:
        ngrok.kill()
        time.sleep(1)
        ngrok.set_auth_token(token)
        tunnel = ngrok.connect(port, bind_tls=True)
        url    = tunnel.public_url
        import httpx
        for attempt in range(8):
            try:
                resp = httpx.get(f"{url}/health", timeout=5)
                if resp.status_code == 200:
                    log_dep.info("ngrok tunnel verified: %s", url)
                    return url
            except Exception:
                pass
            time.sleep(2)
        log_dep.warning("ngrok health check did not respond — returning URL anyway")
        return url
    except Exception as e:
        log_dep.error("ngrok error: %s", e)
        return None


_started = threading.Event()
_failed  = threading.Event()


def _run_server():
    import uvicorn
    cfg = uvicorn.Config(
        app=socket_app,      # FIX #23: socket_app now exists
        host="0.0.0.0",
        port=PORT,
        log_level="warning",
        access_log=False,
        loop="asyncio",
    )
    srv = uvicorn.Server(cfg)

    # Signal the main thread once startup completes
    orig_startup = srv.startup
    async def _patched_startup(sockets=None):
        await orig_startup(sockets)
        _started.set()
    srv.startup = _patched_startup

    try:
        asyncio.run(srv.serve())
    except Exception as e:
        log_dep.error("Server crashed: %s", e)
        _failed.set()


# ── Launch ────────────────────────────────────────────────────
print("=" * 60)
print("  SMART CLASSROOM AI — LAUNCHING")
print("=" * 60)

log_dep.info("Freeing port %d…", PORT)
free_port(PORT)

srv_thread = threading.Thread(target=_run_server, daemon=True, name="uvicorn")
srv_thread.start()

if not _started.wait(30):
    if _failed.is_set():
        raise RuntimeError("Server crashed on startup — check logs above.")
    raise RuntimeError("Server did not start within 30 s — check logs.")

log_dep.info("Server listening on 0.0.0.0:%d", PORT)

if not teaching_engine.is_loaded:
    print("⏳ Waiting for LLM (up to 10 min)…")
    if teaching_engine.wait_until_loaded(600):
        print("✅ LLM ready")
    else:
        print(f"⚠️  LLM timed out — /ask returns 503. Error: {teaching_engine.load_error}")

print("🌐 Starting ngrok tunnel…")
public_url = start_ngrok(PORT)

print("\n" + "=" * 60)
print("  SMART CLASSROOM AI — ONLINE")
print("=" * 60)
if public_url:
    print(f"  Public URL : {public_url}")
    print(f"  API docs   : {public_url}/docs")
    print(f"  Health     : {public_url}/health")
    print(f"  Socket.IO  : {public_url} (ws)")
else:
    print(f"  Local only : http://localhost:{PORT}/docs")
print(f"\n  LLM loaded : {teaching_engine.is_loaded}")
print(f"  STT loaded : {stt_engine.loaded}")
print(f"  Formats    : {list(file_processor.SUPPORTED_FORMATS.keys())}")
print(f"  DB         : {'connected ✅' if supabase_db.connected else 'DISCONNECTED ❌'}")
print("\n  Ctrl+C to stop")
print("=" * 60)

# ── Keep-alive watchdog ───────────────────────────────────────
try:
    while True:
        time.sleep(5)
        if not srv_thread.is_alive():
            log_dep.warning("Server thread died — restarting…")
            _started.clear()
            _failed.clear()
            srv_thread = threading.Thread(target=_run_server, daemon=True, name="uvicorn-restart")
            srv_thread.start()
            if not _started.wait(30):
                raise RuntimeError("Server restart failed.")
            log_dep.info("Server restarted ✅")
except KeyboardInterrupt:
    print("\n⛔ Stopped.")

  SMART CLASSROOM AI — LAUNCHING
⏳ Waiting for LLM (up to 10 min)…
✅ LLM ready
🌐 Starting ngrok tunnel…


/tmp/ipykernel_23863/221405720.py:197: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),



  SMART CLASSROOM AI — ONLINE
  Public URL : https://intersystematic-yolonda-gymnogenous.ngrok-free.dev
  API docs   : https://intersystematic-yolonda-gymnogenous.ngrok-free.dev/docs
  Health     : https://intersystematic-yolonda-gymnogenous.ngrok-free.dev/health
  Socket.IO  : https://intersystematic-yolonda-gymnogenous.ngrok-free.dev (ws)

  LLM loaded : True
  STT loaded : True
  Formats    : ['.pdf', '.docx', '.pptx', '.txt', '.md']
  DB         : connected ✅

  Ctrl+C to stop


# 🛠️ Backend Evaluation & Simulation Suite
This suite runs a series of diagnostic tests against your initialized objects (`auth_manager`, `supabase_db`, `file_processor`, etc.) to verify stability.

In [ ]:
import unittest
import requests
from pathlib import Path

class BackendIntegrityTest(unittest.TestCase):

    def test_01_database_connectivity(self):
        """Verify Supabase and SQLite connections."""
        self.assertTrue(supabase_db.connected(), "Supabase connection failed.")
        self.assertTrue(Path(DIRS['cache'] / 'classroom.db').exists(), "Local SQLite cache file missing.")

    def test_02_auth_cycle(self):
        """Simulate user registration and login."""
        test_user = f"test_{secrets.token_hex(4)}"
        # Test Registration
        uid, err = auth_manager.register_user(test_user, f"{test_user}@test.com", "Password123!", "student")
        self.assertIsNotNone(uid, f"Registration failed: {err}")

        # Test Login
        user, err = auth_manager.authenticate(test_user, "Password123!")
        self.assertIsNotNone(user, f"Login failed: {err}")

        # Test JWT
        token = auth_manager.generate_token(user)
        verify = auth_manager.verify_token(token)
        self.assertTrue(verify.ok, "JWT Verification failed.")

    def test_03_file_processor_logic(self):
        """Test FileProcessor with a dummy text file."""
        test_file = DIRS['uploads'] / "eval_test.txt"
        test_file.write_text("This is a test document for RAG ingestion evaluation.")

        result = file_processor.process_file(test_file)
        self.assertTrue(result['success'], f"File processing failed: {result.get('error')}")
        self.assertIn("evaluation", result['content']['text'])

        if test_file.exists(): test_file.unlink()

    def test_04_api_responsiveness(self):
        """Check if the FastAPI server is actually reachable."""
        try:
            resp = requests.get(f"http://localhost:{PORT}/health", timeout=5)
            self.assertEqual(resp.status_code, 200)
            self.assertEqual(resp.json()['status'], 'ok')
        except Exception as e:
            self.fail(f"API Server is unreachable on port {PORT}: {e}")

def run_evaluation():
    suite = unittest.TestLoader().loadTestsFromTestCase(BackendIntegrityTest)
    unittest.TextTestRunner(verbosity=2).run(suite)

run_evaluation()